## Run Splink on datasets 1 and 2

In [1]:
# Amanda Rodgers
# 2025_02_12

### Import Libraries

In [4]:
import pandas as pd
import splink

### Import datasets

In [6]:
# Load the datasets
dataset_1 = pd.read_csv('datasets/dataset_1.csv')
dataset_2 = pd.read_csv('datasets/dataset_2.csv')

### Create settings, comparison functions and Linker object

In [8]:
import splink.comparison_library as cl

from splink import DuckDBAPI, Linker, SettingsCreator, block_on

settings = SettingsCreator(
    link_type="link_only",
    blocking_rules_to_generate_predictions=[
        block_on("substr(first_name, 1, 2)"),
        block_on("last_name"),
        block_on("email")
    ],
    comparisons=[
        cl.NameComparison(
            "first_name",
        ),
        cl.NameComparison("last_name"),
        cl.ExactMatch("city").configure(term_frequency_adjustments=True),
        cl.EmailComparison("email"),
    ],
)

linker = Linker(
    [dataset_1, dataset_2],
    settings,
    db_api=DuckDBAPI(),
    input_table_aliases=["df_left", "df_right"],
)

## Estimation of probability_two_random_records_match

In [ ]:
#In this example, I guess that the following deterministic matching rules have a recall of about 90%. That means, between them, the rules recover 90% of all true matches.¶
# Note: need more research on how to better estimate the recall

In [13]:
deterministic_rules = [
        block_on("substr(first_name, 1, 2)"),
        block_on("last_name"),
        block_on("email")
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.9)

Probability two random records match is estimated to be  0.0283.
This means that amongst all possible pairwise record comparisons, one in 35.27 are expected to match.  With 1,000,000 total possible comparisons, we expect a total of around 28,348.89 matching pairs


## Estimation of u probabilities

In [15]:
# U-values are part of the Fellegi-Sunter model ( the linkage model Splink runs on), they represent the likelihood that a given attribute agreement
# between two records is due to random chance ( assuming that the records do not refer to the same entity)

In [16]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - last_name (no m values are trained).
    - city (no m values are trained).
    - email (no m values are trained).


## Estimation of m probabilities

In [18]:
# M-values are part of the Fellegi_Sunter model too and they represent the likelihood that a given attribute between two records occurs because 
# the records actually refer to the same entity and not just by chance. It uses an iterative maximum likelihood approach called Expectation Maximisation.

In [19]:
training_blocking_rule = block_on("first_name", "last_name")
training_session_fname_sname = (
    linker.training.estimate_parameters_using_expectation_maximisation(training_blocking_rule)
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."first_name" = r."first_name") AND (l."last_name" = r."last_name")

Parameter estimates will be made for the following comparison(s):
    - city
    - email

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - first_name
    - last_name

Level Exact match on username on comparison email not observed in dataset, unable to train m value

Level Jaro-Winkler distance of email >= 0.88 on comparison email not observed in dataset, unable to train m value

Level Jaro-Winkler >0.88 on username on comparison email not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.0433 in the m_probability of city, level `All other comparisons`
Iteration 2: Largest change in params was 0.00665 in the m_probability of city, level `Exact match on city`
Iteration 3: Largest change in params was 7.34e-05 

In [20]:
# Train more/different possibilities for m-values
training_blocking_rule = block_on("email")
training_session_dob = linker.training.estimate_parameters_using_expectation_maximisation(
    training_blocking_rule
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."email" = r."email"

Parameter estimates will be made for the following comparison(s):
    - first_name
    - last_name
    - city

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - email

Level Jaro-Winkler distance of first_name >= 0.92 on comparison first_name not observed in dataset, unable to train m value

Level Jaro-Winkler distance of first_name >= 0.88 on comparison first_name not observed in dataset, unable to train m value

Level Jaro-Winkler distance of last_name >= 0.92 on comparison last_name not observed in dataset, unable to train m value

Level Jaro-Winkler distance of last_name >= 0.88 on comparison last_name not observed in dataset, unable to train m value

Iteration 1: Largest change in params was 0.0855 in the m_probability of first_name, level `Jaro-Winkler distance of first_name >= 0.7`
Iterati

In [22]:
# Note: need more research on how to better utilize u and m probabilities in splink. There are other algorithms to calculate them too. 

In [23]:
# Save the model and estimated parameters to a .json file
import os
settings = linker.misc.save_model_to_json(
    "./saved_brochman_model.json", overwrite=True
)

### Predicting results/matches and running the model with .predict

In [40]:
# Run model 
# Under the hood this will:
# Generate all pairwise record comparisons that match at least one of the blocking_rules_to_generate_predictions
# Use the rules specified in the Comparisons to evaluate the similarity of the input data
# Use the estimated match weights, applying term frequency adjustments where requested to produce the final match_weight and match_probability scores
from splink import Linker, DuckDBAPI
df_predictions = linker.inference.predict(threshold_match_probability=0.2)
df_predictions.as_pandas_dataframe

Blocking time: 0.06 seconds
Predict time: 0.18 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'first_name':
    m values not fully trained
Comparison: 'last_name':
    m values not fully trained
Comparison: 'email':
    m values not fully trained


<bound method DuckDBDataFrame.as_pandas_dataframe of <splink.internals.duckdb.dataframe.DuckDBDataFrame object at 0x000001D23A753890>>

In [49]:
from splink import Linker, DuckDBAPI
import pandas as pd

# Assuming you have already created and configured the linker object

# Run the inference prediction
df_predictions = linker.inference.predict(threshold_match_probability=0.2)

# Convert the predictions to a Pandas DataFrame
df_predictions_df = df_predictions.as_pandas_dataframe

# Now df_predictions_df is a Pandas DataFrame
print(type(df_predictions))

Blocking time: 0.06 seconds
Predict time: 0.20 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'first_name':
    m values not fully trained
Comparison: 'last_name':
    m values not fully trained
Comparison: 'email':
    m values not fully trained


<class 'splink.internals.duckdb.dataframe.DuckDBDataFrame'>


In [50]:
# Convert the predictions to a Pandas DataFrame
predictions_df = df_predictions.as_pandas_dataframe()

In [51]:
# Show df
predictions_df.head(5)

,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,first_name_l,first_name_r,gamma_first_name,last_name_l,last_name_r,gamma_last_name,city_l,city_r,gamma_city,email_l,email_r,gamma_email,match_key
0,34.612060,1.0,df_left,df_right,221,221,Sierra,Sierra,4,Hines,Hines,4,Morrisborough,Morrisborough,1,millererin@example.net,millererin@example.net,4,0
1,32.612060,1.0,df_left,df_right,342,342,Lucas,Lucas,4,Leon,Leon,4,Christopherstad,Christopherstad,1,zachary79@example.org,zachary79@example.org,4,0
2,33.612060,1.0,df_left,df_right,379,379,Evelyn,Evelyn,4,Mullen,Mullen,4,Craigton,Craigton,1,paulthomas@example.com,paulthomas@example.com,4,0
3,33.612060,1.0,df_left,df_right,610,610,Arthur,Arthur,4,Davidson,Davidson,4,Ruizton,Ruizton,1,deleonholly@example.com,deleonholly@example.com,4,0
4,33.027097,1.0,df_left,df_right,649,649,Isabella,Isabella,4,Wells,Wells,4,Crawfordchester,Crawfordchester,1,matthewbates@example.net,matthewbates@example.net,4,0


In [25]:
# match_weight is a relative measure with no specific range, higher values indicate a higher likelihood that two records are same entity
# match_probability is calculated from match_weight (range 0-1), 0 = records unlikely a match, 1 = records are very likely to be a match
# match_key is used to group and compare records that are likely to match, it is used to reduce the number of comparisons needed

In [52]:
# Save predictions_df to csv file
predictions_df.to_csv('predictions.csv', index=False)

### Evaluation with labeled data

In [53]:
## This is just an example and am going to put in own df_labels after I create it
from splink.datasets import splink_dataset_labels

df_labels = splink_dataset_labels.fake_1000_labels
labels_table = linker.table_management.register_labels_table(df_labels)
df_labels.head(5)

,unique_id_l,source_dataset_l,unique_id_r,source_dataset_r,clerical_match_score
0,0,fake_1000,1,fake_1000,1.0
1,0,fake_1000,2,fake_1000,1.0
2,0,fake_1000,3,fake_1000,1.0
3,0,fake_1000,4,fake_1000,0.0
4,0,fake_1000,5,fake_1000,0.0


In [57]:
# Create ground truth labeled data

# Define the data for the DataFrame
data = {
    'unique_id_l': range(1, 101),
    'source_dataset_l': ['dataset_1'] * 100,
    'unique_id_r': range(1, 101),
    'source_dataset_r': ['dataset_2'] * 100,
    'clerical_match_score': [1] * 100
}

# Create the DataFrame
df_labels = pd.DataFrame(data)

# Display the first 5 rows of the DataFrame
df_labels.head(5)

,unique_id_l,source_dataset_l,unique_id_r,source_dataset_r,clerical_match_score
0,1,dataset_1,1,dataset_2,1
1,2,dataset_1,2,dataset_2,1
2,3,dataset_1,3,dataset_2,1
3,4,dataset_1,4,dataset_2,1
4,5,dataset_1,5,dataset_2,1


In [58]:
# test push to gitlab

### test push